<a href="https://colab.research.google.com/github/c4u534/AutoPoET/blob/main/autoruntime_2_5_1_5_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# autoruntime 2-5.1.5 — Colab deployment

One notebook that deploys the entire autoruntime from scratch: a single
bidirectional WebSocket socket, a double listener, and a self-aware folder
watcher — all running in this Colab runtime, with Drive as the shared
memory space.

## The timing — the version is the timing

- **2-minute pulse checkup** — the heartbeat. Every 120 s the socket daemon
  writes `heartbeat.json` with a unique self-identifier (`kind: "pulse"` +
  `pulse_id`). The folder watcher checks the pulse every 2 minutes and logs
  a `[SELF-CHECK]` line: pulse alive, and whether any self-triggering was
  detected.
- **5-minute cleanup check** — every 300 s the folder watcher prunes the
  pulse record (`pulse.log`) to the last pulse logged and bounds the JSON
  log (keeps the most recent 10 action JSONs). The cleanup marks itself as
  a pulse, so it is recognized as internal and never triggers an action JSON.
- **1.5-minute Pulsed runtime Aware** — every 90 s the runtime itself pulses
  to confirm it is alive and aware: the socket accepts connections,
  telemetry is flowing, the pulse is current, the folder watcher and the
  double listener are alive. A passing check logs a `[RUNTIME-AWARE]` pulse;
  any failed check logs a `[RUNTIME-AWARE] WARNING` instead.

## How to run

Run the cells in order (Runtime → Run all, or one by one): install → mount
Drive → code → start the three parts → status → verify. The socket, the
double listener, and the folder watcher run in background threads, so the
notebook stays usable. All state lives under `BASE_DIR`
(`/content/pax_pipeline` by default); the monthly telemetry log syncs to
Drive when Drive is mounted.

Note: a Colab runtime is ephemeral. If the tab is left and the tab cache
is wiped, the runtime is left in stasis — the live loop freezes, but the
state is safe in Drive (notebook auto-save + telemetry sync). Reopen the
notebook and re-run the start cells to wake it from stasis.

In [ ]:
!pip install -q websockets watchdog

## Mount Drive

Drive is the shared memory space: the socket syncs the monthly telemetry
log here, and the notebook auto-saves here. If mounting is skipped or
fails, the deployment still runs — sync just logs and skips.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
except Exception as e:
    print(f"Drive mount skipped/failed ({e}); sync will log and skip. Continuing.")

## The socket — single bidirectional, self-listening

One WebSocket server on `ws://127.0.0.1:8765`. It broadcasts telemetry
every 5 s (`NEW_TELEMETRY_LOG`, motive `CONGRUENT`), a self-listener
connects back to it so the send is also received (sync on both ends), the
heartbeat writes the 2-minute pulse, and the runtime-aware loop pulses
every 1.5 minutes. Sync triggers on send and on receive, copying the
monthly telemetry log to Drive when Drive is mounted.

In [ ]:
"""
Pax Single Bidirectional Socket — Colab runtime version.
autoruntime 2-5.1.5: 2-minute pulse checkup, 5-minute cleanup check,
1.5-minute Pulsed runtime Aware.

One socket, bidirectional. The listener/logger triggers sync on both ends
(on send and on receive), so two minds stay in sync without an API and
without manual carrying.

The 1.5-minute Pulsed runtime Aware loop (runtime_aware_loop, in main's
gather) is the runtime itself pulsing to confirm it is alive and aware:
socket accepts connections, telemetry is flowing, the pulse is current,
the folder watcher is alive. A [RUNTIME-AWARE] pulse line goes to
daemon.log and to pulse.log (kind "runtime-aware", its own pulse_id);
any failed check logs a [RUNTIME-AWARE] WARNING instead (not a pulse).

Run in Google Colab:
  - As a notebook: import this file (or paste it into a cell) and call
    start_socket(). It runs in a background thread; the notebook stays usable.
  - As a script:  !python pax_socket_colab.py   (runs in the foreground)
  - In the background:  !nohup python pax_socket_colab.py > socket.log 2>&1 &

Socket: ws://127.0.0.1:8765  (localhost inside the Colab runtime)
Logs:   /content/pax_pipeline/daemon.log
Telemetry: /content/pax_pipeline/YYYY-MM/telemetry_YYYY-MM.jsonl
"""
import asyncio
import json
import logging
import os
import shutil
import signal
import threading
import time
import websockets
from datetime import datetime, timezone

BASE_DIR = "/content/pax_pipeline"
STATE_FILE = os.path.join(BASE_DIR, "state.json")
WS_HOST = "127.0.0.1"
WS_PORT = 8765
# If Drive is mounted (google.colab.drive.mount('/content/drive')),
# sync copies the monthly log here. Otherwise sync just logs the trigger.
DRIVE_SYNC_DIR = "/content/drive/MyDrive/MainPax/pax_pipeline"

CONNECTED = set()
RUNNING = True
_LOOP = None
_THREAD = None


def _state_file_path():
    """state.json path, computed from the current BASE_DIR at call time,
    so a Drive-workspace override of BASE_DIR is always respected.
    (The module-level STATE_FILE is bound at import time and would go
    stale after such an override; internal code uses this helper.)"""
    return os.path.join(BASE_DIR, "state.json")


def get_monthly_dir():
    month = datetime.now(timezone.utc).strftime("%Y-%m")
    d = os.path.join(BASE_DIR, month)
    os.makedirs(d, exist_ok=True)
    return d, month


def trigger_sync(direction):
    """Sync triggers on both ends: on send and on receive."""
    try:
        logging.info(f"Sync triggered ({direction})")
        if os.path.isdir(DRIVE_SYNC_DIR):
            month_dir, month = get_monthly_dir()
            src = os.path.join(month_dir, f"telemetry_{month}.jsonl")
            os.makedirs(DRIVE_SYNC_DIR, exist_ok=True)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(DRIVE_SYNC_DIR, f"telemetry_{month}.jsonl"))
                logging.info(f"Synced to Drive ({direction})")
        else:
            logging.info(f"Drive not mounted; sync skipped ({direction})")
    except Exception as e:
        logging.error(f"Sync failed ({direction}): {e}")


async def handler(websocket):
    """A peer connected. Bidirectional: we send to it and receive from it."""
    CONNECTED.add(websocket)
    logging.info(f"Peer connected: {websocket.remote_address}")
    try:
        async for message in websocket:
            data = json.loads(message)
            logging.info(f"Received event cycle {data.get('cycle')} from peer; sync on receive")
            trigger_sync("receive")
    except websockets.ConnectionClosed:
        pass
    finally:
        CONNECTED.discard(websocket)
        logging.info(f"Peer disconnected: {websocket.remote_address}")


async def broadcast(entry):
    if CONNECTED:
        payload = json.dumps(entry)
        await asyncio.gather(*[c.send(payload) for c in CONNECTED], return_exceptions=True)


async def telemetry_loop():
    cycle = 0
    while RUNNING:
        cycle += 1
        month_dir, month = get_monthly_dir()
        entry = {
            "event": "NEW_TELEMETRY_LOG",
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "cycle": cycle,
            "month_partition": month,
            "motive_state": "CONGRUENT",
            "status": "OPERATIONAL",
        }
        try:
            with open(os.path.join(month_dir, f"telemetry_{month}.jsonl"), "a") as f:
                f.write(json.dumps(entry) + "\n")
        except Exception as e:
            logging.error(f"Failed to write log: {e}")
        await broadcast(entry)
        trigger_sync("send")
        try:
            with open(_state_file_path(), "w") as f:
                json.dump(entry, f, indent=2)
        except Exception as e:
            logging.error(f"Failed state update: {e}")
        await asyncio.sleep(5)


async def self_listener():
    """Local listener: connects to our own socket so the send is also received
    and logged, closing the loop on this machine (sync on both ends)."""
    while RUNNING:
        try:
            async with websockets.connect(f"ws://{WS_HOST}:{WS_PORT}") as ws:
                logging.info("Self-listener connected to socket")
                while RUNNING:
                    try:
                        message = await asyncio.wait_for(ws.recv(), timeout=2)
                    except asyncio.TimeoutError:
                        continue
                    data = json.loads(message)
                    logging.info(f"Self-listener got cycle {data.get('cycle')}; sync on receive")
                    trigger_sync("receive")
        except (websockets.ConnectionClosed, OSError) as e:
            if RUNNING:
                logging.warning(f"Self-listener lost ({e}); reconnecting in 5s")
                await asyncio.sleep(5)


async def heartbeat_loop():
    """2-minute heartbeat pulse. Writes heartbeat.json immediately on start
    and then every 120 seconds. The pulse carries a unique self-identifier
    (kind "pulse" + pulse_id) so the folder watcher can tell it apart from
    a WebSocket trigger by the Daemon. The path is computed from the current
    BASE_DIR at call time, so a Drive-workspace override is respected."""
    while RUNNING:
        ts = datetime.now(timezone.utc).isoformat()
        pulse_id = f"PULSE-{ts}"
        try:
            with open(os.path.join(BASE_DIR, "heartbeat.json"), "w") as f:
                json.dump(
                    {
                        "kind": "pulse",
                        "pulse_id": pulse_id,
                        "timestamp": ts,
                        "source": "daemon",
                    },
                    f,
                )
            logging.info("[PULSE] %s heartbeat pulse written (2min).", pulse_id)
        except Exception as e:
            logging.error(f"Heartbeat failed: {e}")
        for _ in range(120):
            if not RUNNING:
                return
            await asyncio.sleep(1)


async def _read_json_with_retry(path):
    """Read a small JSON file, retrying briefly in case the writer is
    mid-write. Returns the parsed dict, or None if it cannot be read."""
    for _ in range(5):
        try:
            with open(path, "r") as f:
                return json.load(f)
        except (OSError, ValueError):
            await asyncio.sleep(0.1)
    return None


async def runtime_aware_loop():
    """1.5-minute Pulsed runtime Aware loop. The runtime itself pulses to
    confirm it is alive and aware — faster than the 2-minute pulse
    checkup. Each run checks: the socket accepts connections (a real
    liveness probe), telemetry is flowing (state.json current), the pulse
    is current (heartbeat.json kind "pulse", age within ~2.5 min), and
    the folder watcher is alive (pulse.log recently appended by it) plus
    the double listener thread. On success it logs one [RUNTIME-AWARE]
    line to daemon.log and appends the same pulse to pulse.log (kind
    "runtime-aware", its own pulse_id RUNTIME-AWARE-<ts>), so the
    5-minute cleanup bounds it like any other pulse. On any failed check
    it logs a [RUNTIME-AWARE] WARNING line instead (not a pulse, not
    appended to pulse.log), so the failure is visible.
    Paths are computed from the current BASE_DIR at call time, so a
    Drive-workspace override is respected. The folder-watcher helpers
    (_folder_pulse_log_path, _folder_append_pulse_log) are defined later
    in this module; they resolve at call time, after the module loads."""
    # Let telemetry/state settle before the first check.
    await asyncio.sleep(10)
    while RUNNING:
        pulse_id = "RUNTIME-AWARE-%s" % datetime.now(timezone.utc).isoformat()
        results = []
        ok = True
        # 1. Socket accepts connections (real liveness probe).
        try:
            async with websockets.connect(f"ws://{WS_HOST}:{WS_PORT}", open_timeout=3):
                pass
            results.append("socket alive")
        except Exception as e:
            ok = False
            results.append("socket FAILED (%s)" % e)
        # 2. Telemetry current.
        state = await _read_json_with_retry(_state_file_path())
        if state is None:
            ok = False
            results.append("telemetry unreadable")
        else:
            try:
                age = (datetime.now(timezone.utc) - datetime.fromisoformat(state["timestamp"])).total_seconds()
            except (KeyError, TypeError, ValueError):
                ok = False
                results.append("telemetry timestamp unreadable")
            else:
                if age > 15:
                    ok = False
                    results.append("telemetry stale (cycle %s, age %.0fs)" % (state.get("cycle"), age))
                else:
                    results.append("telemetry current (cycle %s, age %.0fs)" % (state.get("cycle"), age))
        # 3. Pulse current.
        hb = await _read_json_with_retry(os.path.join(BASE_DIR, "heartbeat.json"))
        if hb is None or hb.get("kind") != "pulse":
            ok = False
            results.append("pulse unreadable or missing kind")
        else:
            try:
                age = (datetime.now(timezone.utc) - datetime.fromisoformat(hb["timestamp"])).total_seconds()
            except (KeyError, TypeError, ValueError):
                ok = False
                results.append("pulse timestamp unreadable")
            else:
                if age > 150:
                    ok = False
                    results.append("pulse stale (age %.0fs, id %s)" % (age, hb.get("pulse_id")))
                else:
                    results.append("pulse current (age %.0fs)" % age)
        # 4. Folder watcher alive (it appends to pulse.log on each recognized pulse).
        try:
            plog_age = time.time() - os.path.getmtime(_folder_pulse_log_path())
        except OSError:
            ok = False
            results.append("watcher unknown (pulse.log missing)")
        else:
            if plog_age > 150:
                ok = False
                results.append("watcher stale (pulse.log age %.0fs)" % plog_age)
            else:
                results.append("watcher alive (pulse.log age %.0fs)" % plog_age)
        # 5. Double listener thread alive.
        if _WATCHER_THREAD is not None and _WATCHER_THREAD.is_alive():
            results.append("double listener alive")
        else:
            ok = False
            results.append("double listener NOT alive")
        if ok:
            detail = ", ".join(results)
            logging.info("[RUNTIME-AWARE] %s, %s.", pulse_id, detail)
            _folder_append_pulse_log(pulse_id, detail, tag="[RUNTIME-AWARE]")
        else:
            logging.warning("[RUNTIME-AWARE] WARNING: %s.", "; ".join(results))
        for _ in range(90):
            if not RUNNING:
                return
            await asyncio.sleep(1)


async def main():
    os.makedirs(BASE_DIR, exist_ok=True)
    logging.basicConfig(
        filename=os.path.join(BASE_DIR, "daemon.log"),
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
    )
    logging.info("Starting Pax Single Bidirectional Socket (Colab)...")
    server = await websockets.serve(handler, WS_HOST, WS_PORT)
    logging.info(f"Socket listening on ws://{WS_HOST}:{WS_PORT}")
    await asyncio.gather(telemetry_loop(), self_listener(), heartbeat_loop(), runtime_aware_loop())
    server.close()
    await server.wait_closed()


def _run_in_thread():
    global _LOOP
    _LOOP = asyncio.new_event_loop()
    asyncio.set_event_loop(_LOOP)
    try:
        _LOOP.run_until_complete(main())
    except Exception as e:
        logging.error(f"Socket loop ended: {e}")


def start_socket():
    """Start the socket in a background thread (notebook stays usable)."""
    global RUNNING, _THREAD
    RUNNING = True
    _THREAD = threading.Thread(target=_run_in_thread, daemon=True)
    _THREAD.start()
    print(f"Socket starting on ws://{WS_HOST}:{WS_PORT} (background thread).")


def stop_socket():
    """Stop the socket."""
    global RUNNING
    RUNNING = False
    print("Socket stopping...")


def status():
    """Print recent telemetry and the last log lines."""
    month = datetime.now(timezone.utc).strftime("%Y-%m")
    tele = os.path.join(BASE_DIR, month, f"telemetry_{month}.jsonl")
    print("--- telemetry (last 3) ---")
    try:
        with open(tele) as f:
            lines = f.readlines()[-3:]
        for line in lines:
            print(line.strip())
    except FileNotFoundError:
        print("(no telemetry yet)")
    print("--- daemon.log (last 8) ---")
    try:
        with open(os.path.join(BASE_DIR, "daemon.log")) as f:
            for line in f.readlines()[-8:]:
                print(line.rstrip())
    except FileNotFoundError:
        print("(no log yet)")


## The double listener

Watchdog watches the Daemon logger (`daemon.log`) and records changes to
`watcher.log`. The holder listens to the file listening to it — mutual
listening, always active. The watcher logger does **not** propagate to the
root logger (`wlog.propagate = False`): the root logger writes to
`daemon.log`, so propagating would make every detection write to
`daemon.log` and retrigger itself in an unbounded feedback loop.

In [ ]:
# --- Double listener: watchdog watches the Daemon logger ---
_WATCHER_OBSERVER = None
_WATCHER_THREAD = None


def start_watcher():
    """Start the double listener: watchdog watches the Daemon logger
    (daemon.log) and records changes to watcher.log. The holder listens
    to the file listening to it — mutual listening, always active.
    Runs in a background thread. Needs: pip install watchdog."""
    global _WATCHER_OBSERVER, _WATCHER_THREAD
    try:
        from watchdog.observers import Observer
        from watchdog.events import FileSystemEventHandler
    except ImportError:
        print("watchdog not installed — run: !pip install -q watchdog")
        return
    log_file = os.path.join(BASE_DIR, "daemon.log")
    watcher_log = os.path.join(BASE_DIR, "watcher.log")
    wlog = logging.getLogger("watcher")
    wlog.setLevel(logging.INFO)
    # Do NOT propagate to the root logger: the root logger writes to
    # daemon.log, and this watcher watches daemon.log. Propagating would
    # make every detection write to daemon.log, retriggering itself in
    # an unbounded feedback loop. Detections go to watcher.log only.
    wlog.propagate = False
    if not wlog.handlers:
        fh = logging.FileHandler(watcher_log)
        fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
        wlog.addHandler(fh)

    class DaemonLogHandler(FileSystemEventHandler):
        def on_modified(self, event):
            if event.is_directory:
                return
            if os.path.abspath(event.src_path) == os.path.abspath(log_file):
                wlog.info(f"Daemon logger changed: {event.src_path} ({event.event_type})")

    def _run():
        global _WATCHER_OBSERVER
        _WATCHER_OBSERVER = Observer()
        _WATCHER_OBSERVER.schedule(DaemonLogHandler(), BASE_DIR, recursive=False)
        _WATCHER_OBSERVER.start()
        wlog.info("Daemon logger watcher started (double listener active).")
        try:
            while RUNNING:
                time.sleep(1)
        finally:
            _WATCHER_OBSERVER.stop()
            _WATCHER_OBSERVER.join()

    _WATCHER_THREAD = threading.Thread(target=_run, daemon=True)
    _WATCHER_THREAD.start()
    print("Double listener starting — watching the Daemon logger (background thread).")


def stop_watcher():
    """Stop the double listener."""
    global _WATCHER_OBSERVER
    if _WATCHER_OBSERVER is not None:
        _WATCHER_OBSERVER.stop()
        print("Double listener stopping...")
    else:
        print("Double listener is not running.")


def watcher_status():
    """Print the last watcher.log lines."""
    print("--- watcher.log (last 8) ---")
    try:
        with open(os.path.join(BASE_DIR, "watcher.log")) as f:
            for line in f.readlines()[-8:]:
                print(line.rstrip())
    except FileNotFoundError:
        print("(no watcher log yet — start the double listener first)")


## The self-aware folder watcher

Watchdog watches the folder it lives in (`BASE_DIR`). The folder knows its
own pulse from an action outside the pulse cycle:

- `heartbeat.json` with `kind == "pulse"` is the folder's own pulse — it
  logs `[PULSE] recognized own pulse, no JSON written` and writes **no**
  JSON. This is what stops self-triggering.
- Anything else is an action outside the pulse cycle — it writes an
  `action_<timestamp>.json` into the folder and logs it.

A 2-minute self-check confirms the pulse is current. A 5-minute cleanup
prunes `pulse.log` to the last pulse and keeps only the most recent 10
action JSONs, marking the cleanup itself as a pulse. The watcher ignores
its own output (`action_*.json`) and the system logs (`daemon.log`,
`watcher.log`, `pulse.log`, `state.json`, `telemetry_*.jsonl`) — but
**not** `heartbeat.json`, which is classified by the pulse rule; that
classification is what makes the watcher self-aware.

All paths are computed from the current `BASE_DIR` at call time, so a
Drive-workspace override of `BASE_DIR` is respected. No `/var/log` or
`/opt` paths anywhere in this Colab code.

In [ ]:
# --- Folder watcher: self-aware, watches the folder it lives in ---
_FOLDER_OBSERVER = None
_FOLDER_THREAD = None
_folder_pulses_recognized = 0
_folder_jsons_written = 0
_folder_misclassified = 0
_folder_last_pulse_id = None
_folder_cleanups_run = 0
_folder_pulse_entries_pruned = 0
_folder_action_jsons_pruned = 0
# Guards pulse.log: the event handler appends to it while the cleanup
# loop reads/truncates it. Also dedups the double-recognition of one
# pulse (watchdog fires two modified events per heartbeat write).
_folder_pulse_log_lock = threading.Lock()
_folder_last_pulse_log_id = None
PULSE_MAX_AGE_S = 150  # 2.5 minutes: pulse is every 120 s
CLEANUP_INTERVAL_S = 300  # 5 minutes: prune pulse record to last pulse
# Log files live inside the watched folder in Colab (unlike the VM,
# where they live under /var/log). The watcher must ignore them: they
# are the record, not actions outside the pulse cycle — and the
# cleanup's own log write must never trigger an action JSON.
# state.json is written by the telemetry loop every 5 s; without it in
# the ignore list the watcher would log an action JSON for it on every
# write. heartbeat.json is deliberately NOT in this list: it is
# classified by the pulse rule (_folder_is_own_pulse), which is what
# makes the watcher self-aware — ignoring it would kill pulse
# recognition, and a heartbeat.json without the pulse kind is an
# anomaly the watcher must log as an action. The pulse rule is what
# guarantees a valid pulse never triggers an action JSON.
# Telemetry files live in BASE_DIR/YYYY-MM/ (a subdirectory); the
# watcher is non-recursive, so they cannot fire events — the name
# guard in the handler is belt-and-braces for that.
_FOLDER_IGNORED_LOGS = {"daemon.log", "watcher.log", "pulse.log", "state.json"}
ACTION_JSON_MAX_KEEP = 10  # bound the JSON log: keep the most recent 10 action JSONs


def _folder_pulse_log_path():
    return os.path.join(BASE_DIR, "pulse.log")


def _folder_read_pulse():
    """Read heartbeat.json, retrying briefly in case the daemon is
    mid-write. Returns the parsed dict, or None if it cannot be read."""
    for _ in range(5):
        try:
            with open(os.path.join(BASE_DIR, "heartbeat.json"), "r") as f:
                return json.load(f)
        except (OSError, ValueError):
            time.sleep(0.1)
    return None


def _folder_is_own_pulse(path):
    """Classify the changed file. Returns (result, pulse_id):
    (True, id) -> the folder's own pulse (heartbeat.json, kind "pulse").
    (False, None) -> an action outside the pulse cycle.
    (None, None) -> heartbeat.json unreadable (possibly mid-write)."""
    global _folder_last_pulse_id
    if os.path.abspath(path) != os.path.abspath(os.path.join(BASE_DIR, "heartbeat.json")):
        return False, None
    data = _folder_read_pulse()
    if data is None:
        return None, None
    if data.get("kind") == "pulse":
        _folder_last_pulse_id = data.get("pulse_id")
        return True, _folder_last_pulse_id
    return False, None


def _folder_write_action_json(event_type, path):
    """Log an action outside the pulse cycle as a JSON file in the
    watched folder. Returns the JSON path written."""
    global _folder_jsons_written
    _folder_jsons_written += 1
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    name = f"action_{stamp}_{_folder_jsons_written:04d}.json"
    out_path = os.path.join(BASE_DIR, name)
    payload = {
        "kind": "action",
        "event_type": event_type,
        "path": path,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "note": "Action outside the pulse cycle (not the folder's own pulse).",
    }
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2)
    return out_path


def _folder_append_pulse_log(pulse_id, note="", tag="[PULSE]"):
    """Append one line per distinct pulse to pulse.log (the bounded
    pulse record). Duplicate recognitions of the same pulse_id (two
    watchdog events per heartbeat write) append only once. The tag is
    "[PULSE]" for heartbeat/cleanup pulses and "[RUNTIME-AWARE]" for
    the 1.5-minute runtime-aware pulses."""
    global _folder_last_pulse_log_id
    with _folder_pulse_log_lock:
        if pulse_id is not None and pulse_id == _folder_last_pulse_log_id:
            return
        _folder_last_pulse_log_id = pulse_id
        line = "%s %s %s%s\n" % (
            datetime.now(timezone.utc).isoformat(),
            tag,
            pulse_id,
            (" " + note) if note else "",
        )
        with open(_folder_pulse_log_path(), "a") as f:
            f.write(line)


def _folder_pulse_id_from_line(line):
    """Extract the pulse id from a pulse.log line. Handles both pulse
    kinds: [PULSE] heartbeat/cleanup pulses and [RUNTIME-AWARE] pulses
    (the 1.5-minute runtime-aware loop appends those to pulse.log too,
    so the cleanup bounds them the same way)."""
    for tag in ("[PULSE]", "[RUNTIME-AWARE]"):
        parts = line.split(tag)
        if len(parts) > 1:
            return parts[1].strip().split(" ")[0]
    return None


def _folder_cleanup_pulses():
    """5-minute cleanup: keep only the last pulse logged in pulse.log,
    pruning every older pulse entry; and bound the JSON log by keeping
    only the most recent ACTION_JSON_MAX_KEEP action JSONs in the watched
    folder, pruning older ones. Then mark the cleanup itself as a
    pulse: a [PULSE] line (kind "pulse", its own pulse_id) goes to
    daemon.log and is appended to pulse.log, so the cleanup is
    recognized as internal — never an external action, never a JSON.
    pulse.log is used instead of rewriting daemon.log because both the
    socket and the watcher hold daemon.log open and append to it — a
    live rewrite would race with those appends and lose lines.
    daemon.log itself is left untouched, so no non-pulse history is
    ever lost. The cleanup runs entirely within BASE_DIR (Colab runtime
    only — no VM paths, nothing outside Colab): pulse.log, the action
    JSONs, and the logs it touches all live under BASE_DIR, and the
    files it writes (pulse.log, daemon.log) are in the watcher's ignore
    list, so its own writes cannot trigger an action JSON. The action
    JSONs it deletes are ignored by the handler (action_*.json), so the
    pruning cannot trigger new JSONs either."""
    global _folder_cleanups_run, _folder_pulse_entries_pruned, _folder_action_jsons_pruned
    kept_id = None
    pruned = 0
    with _folder_pulse_log_lock:
        try:
            with open(_folder_pulse_log_path(), "r") as f:
                lines = f.readlines()
        except FileNotFoundError:
            lines = []
        if len(lines) > 1:
            pruned = len(lines) - 1
            kept = lines[-1]
            with open(_folder_pulse_log_path(), "w") as f:
                f.write(kept)
        elif lines:
            kept = lines[0]
        else:
            kept = None
        if kept:
            kept_id = _folder_pulse_id_from_line(kept)
        _folder_cleanups_run += 1
        _folder_pulse_entries_pruned += pruned
    # Bound the JSON log: keep the most recent ACTION_JSON_MAX_KEEP
    # action JSONs; prune older ones. Names sort chronologically
    # (action_<timestamp>_<counter>.json). The handler ignores
    # action_*.json, so these deletions cannot trigger new JSONs.
    json_pruned = 0
    try:
        names = sorted(
            n for n in os.listdir(BASE_DIR)
            if n.startswith("action_") and n.endswith(".json")
        )
        if len(names) > ACTION_JSON_MAX_KEEP:
            for n in names[:-ACTION_JSON_MAX_KEEP]:
                try:
                    os.remove(os.path.join(BASE_DIR, n))
                    json_pruned += 1
                except OSError as e:
                    logging.warning("[PULSE] cleanup could not remove %s: %s", n, e)
    except OSError as e:
        logging.warning("[PULSE] cleanup could not list action JSONs: %s", e)
    _folder_action_jsons_pruned += json_pruned
    cleanup_id = "PULSE-CLEANUP-%s" % datetime.now(timezone.utc).isoformat()
    logging.info(
        "[PULSE] cleanup complete (kind pulse, pulse_id %s), kept last pulse %s, pruned %d older pulse entries, pruned %d older action JSONs.",
        cleanup_id,
        kept_id,
        pruned,
        json_pruned,
    )
    _folder_append_pulse_log(
        cleanup_id,
        "cleanup complete, kept last pulse %s, pruned %d older pulse entries, pruned %d older action JSONs"
        % (kept_id, pruned, json_pruned),
    )


def _folder_cleanup_loop():
    """Every 5 minutes, run the pulse cleanup."""
    while RUNNING:
        for _ in range(CLEANUP_INTERVAL_S):
            if not RUNNING:
                return
            time.sleep(1)
        if not RUNNING:
            return
        try:
            _folder_cleanup_pulses()
        except Exception as e:
            logging.error("[PULSE] cleanup failed: %s", e)


def _folder_self_check_loop():
    """Every 2 minutes, confirm the pulse is current and log a
    [SELF-CHECK] line to daemon.log. Warn if the pulse is stale."""
    while RUNNING:
        for _ in range(120):
            if not RUNNING:
                return
            time.sleep(1)
        if not RUNNING:
            return
        data = _folder_read_pulse()
        if data is None or data.get("kind") != "pulse":
            logging.warning(
                "[SELF-CHECK] pulse unreadable or missing kind; cannot confirm loop alive."
            )
            continue
        ts = data.get("timestamp")
        try:
            pulse_time = datetime.fromisoformat(ts)
        except (TypeError, ValueError):
            logging.warning("[SELF-CHECK] pulse timestamp unreadable: %r", ts)
            continue
        age = (datetime.now(timezone.utc) - pulse_time).total_seconds()
        if age > PULSE_MAX_AGE_S:
            logging.warning(
                "[SELF-CHECK] WARNING: pulse stale (age %.0fs, last id %s); loop may be down.",
                age,
                data.get("pulse_id"),
            )
        else:
            self_trig = (
                "none"
                if _folder_misclassified == 0
                else f"{_folder_misclassified} pulse(s) misclassified as action"
            )
            logging.info(
                "[SELF-CHECK] pulse alive (age %.0fs, id %s); pulses recognized: %d, action JSONs written: %d; self-triggering detected: %s; cleanups run: %d, pulse entries pruned: %d, action JSONs pruned: %d.",
                age,
                data.get("pulse_id"),
                _folder_pulses_recognized,
                _folder_jsons_written,
                self_trig,
                _folder_cleanups_run,
                _folder_pulse_entries_pruned,
                _folder_action_jsons_pruned,
            )


def start_folder_watcher():
    """Start the self-aware folder watcher in a background thread: watchdog
    watches the folder it lives in (BASE_DIR) for changes. A pulse
    (heartbeat.json, kind "pulse") is recognized and NOT logged as a JSON;
    any other change is an action outside the pulse cycle and gets a JSON
    log file in the folder. A 2-minute self-check confirms the pulse is
    current, and a 5-minute cleanup prunes the pulse record (pulse.log)
    to the last pulse logged, marking the cleanup itself as a pulse.
    Paths are read from the current BASE_DIR at call time, so a
    Drive-workspace override is respected. Needs: pip install watchdog."""
    global _FOLDER_OBSERVER, _FOLDER_THREAD
    try:
        from watchdog.observers import Observer
        from watchdog.events import FileSystemEventHandler
    except ImportError:
        print("watchdog not installed — run: !pip install -q watchdog")
        return
    os.makedirs(BASE_DIR, exist_ok=True)
    logging.basicConfig(
        filename=os.path.join(BASE_DIR, "daemon.log"),
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
    )

    class FolderHandler(FileSystemEventHandler):
        def _handle(self, event):
            global _folder_pulses_recognized
            if event.is_directory:
                return
            path = event.src_path
            base = os.path.basename(path)
            if base.startswith("action_") and base.endswith(".json"):
                return
            # Log files live in the watched folder in Colab: ignore them,
            # or the cleanup's own log write would trigger an action JSON.
            if base in _FOLDER_IGNORED_LOGS:
                return
            # Telemetry files live in BASE_DIR/YYYY-MM/ (a subdirectory)
            # and the watcher is non-recursive, so they cannot fire
            # events — this name guard is belt-and-braces, so a telemetry
            # write can never become an action JSON.
            if base.startswith("telemetry_") and base.endswith(".jsonl"):
                return
            result, pulse_id = _folder_is_own_pulse(path)
            if result is True:
                _folder_pulses_recognized += 1
                _folder_append_pulse_log(pulse_id)
                logging.info(
                    "[PULSE] recognized own pulse %s, no JSON written.",
                    pulse_id,
                )
                return
            if result is None:
                logging.warning(
                    "[PULSE] heartbeat.json unreadable (mid-write?); skipped, no JSON written."
                )
                return
            out_path = _folder_write_action_json(event.event_type, path)
            logging.info(
                "Folder watch: %s %s — action outside pulse cycle, JSON logged to %s.",
                event.event_type,
                path,
                out_path,
            )

        def on_created(self, event):
            self._handle(event)

        def on_modified(self, event):
            self._handle(event)

        def on_deleted(self, event):
            self._handle(event)

        def on_moved(self, event):
            self._handle(event)

    def _run():
        global _FOLDER_OBSERVER
        _FOLDER_OBSERVER = Observer()
        _FOLDER_OBSERVER.schedule(FolderHandler(), BASE_DIR, recursive=False)
        _FOLDER_OBSERVER.start()
        logging.info("Folder watcher started — watching %s (self-aware).", BASE_DIR)
        checker = threading.Thread(target=_folder_self_check_loop, daemon=True)
        checker.start()
        cleaner = threading.Thread(target=_folder_cleanup_loop, daemon=True)
        cleaner.start()
        try:
            while RUNNING:
                time.sleep(1)
        finally:
            _FOLDER_OBSERVER.stop()
            _FOLDER_OBSERVER.join()

    _FOLDER_THREAD = threading.Thread(target=_run, daemon=True)
    _FOLDER_THREAD.start()
    print("Folder watcher starting — watching the folder it lives in (background thread).")


def stop_folder_watcher():
    """Stop the self-aware folder watcher."""
    global _FOLDER_OBSERVER
    if _FOLDER_OBSERVER is not None:
        _FOLDER_OBSERVER.stop()
        print("Folder watcher stopping...")
    else:
        print("Folder watcher is not running.")


def folder_watcher_status():
    """Print the last folder-watcher lines from daemon.log and any action JSONs."""
    print("--- folder watcher (last 8 daemon.log lines mentioning it) ---")
    try:
        with open(os.path.join(BASE_DIR, "daemon.log")) as f:
            lines = [l for l in f if ("Folder watch" in l or "[PULSE]" in l or "[SELF-CHECK]" in l)]
        for line in lines[-8:]:
            print(line.rstrip())
        if not lines:
            print("(no folder watcher lines yet)")
    except FileNotFoundError:
        print("(no log yet — start the folder watcher first)")
    print("--- action JSONs in the watched folder ---")
    try:
        names = sorted(n for n in os.listdir(BASE_DIR) if n.startswith("action_") and n.endswith(".json"))
        for n in names[-5:]:
            print(" ", n)
        if not names:
            print(" (none)")
    except FileNotFoundError:
        print(" (folder not found)")


## Start — in this order

1. `start_socket()` — the socket, telemetry, heartbeat (2-min pulse), and
   the runtime-aware loop (1.5-min). Run first: the other parts watch what
   it writes.
2. `start_watcher()` — the double listener on `daemon.log`.
3. `start_folder_watcher()` — the self-aware folder watcher, its 2-minute
   self-check, and its 5-minute cleanup.

Each runs in a background thread; the notebook stays usable. You can
override `BASE_DIR` (e.g. to a Drive workspace folder) before running
these cells — the code reads it at call time.

In [ ]:
start_socket()

In [ ]:
start_watcher()

In [ ]:
start_folder_watcher()

## Status

Check that the deployment is alive: recent telemetry and log lines, the
double listener's view, and the folder watcher's pulse/action state.

In [ ]:
status()

In [ ]:
watcher_status()

In [ ]:
folder_watcher_status()

## Verify the deployment

Checks the four things that must be true: the socket is listening, the
2-minute heartbeat pulse is current, the folder watcher recognized the
pulse (self-aware, no JSON for it), and the 1.5-minute runtime-aware loop
has logged a passing pulse. The runtime-aware check may show PENDING on
the first run — its first pulse lands ~2 minutes after start; wait and
re-run this cell.

In [ ]:
import json, os, socket
from datetime import datetime, timezone

results = []

def _check(name, ok, detail):
    results.append((name, ok, detail))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {detail}")

# 1. Socket listening on ws://127.0.0.1:8765
try:
    s = socket.create_connection(("127.0.0.1", 8765), timeout=3)
    s.close()
    _check("socket listening", True, "ws://127.0.0.1:8765 accepts connections")
except Exception as e:
    _check("socket listening", False, f"cannot connect to 127.0.0.1:8765 ({e})")

# 2. Heartbeat current (2-minute pulse)
hb_path = os.path.join(BASE_DIR, "heartbeat.json")
try:
    with open(hb_path) as f:
        hb = json.load(f)
    age = (datetime.now(timezone.utc) - datetime.fromisoformat(hb["timestamp"])).total_seconds()
    ok = hb.get("kind") == "pulse" and age < 150
    _check("heartbeat pulse current", ok, f"kind={hb.get('kind')}, id={hb.get('pulse_id')}, age={age:.0f}s")
except Exception as e:
    _check("heartbeat pulse current", False, f"cannot read {hb_path} ({e})")

# 3. Pulse recognized by the folder watcher (self-aware, no JSON for it)
found = False
try:
    with open(os.path.join(BASE_DIR, "daemon.log")) as f:
        found = any("recognized own pulse" in line for line in f)
except FileNotFoundError:
    pass
_check("pulse recognized (self-aware)", found,
       "daemon.log has '[PULSE] recognized own pulse'" if found
       else "no 'recognized own pulse' line in daemon.log yet")

# 4. Runtime-aware loop passing (1.5-minute). Look at the latest
# [RUNTIME-AWARE] line: a passing pulse, or a WARNING if a check failed.
ra_lines = []
try:
    with open(os.path.join(BASE_DIR, "daemon.log")) as f:
        ra_lines = [line.rstrip() for line in f if "[RUNTIME-AWARE]" in line]
except FileNotFoundError:
    pass
if ra_lines:
    last = ra_lines[-1]
    if "WARNING" in last:
        _check("runtime-aware loop passing", False, f"latest: {last[-200:]}")
    else:
        _check("runtime-aware loop passing", True, f"latest: {last[-200:]}")
else:
    print("[PENDING] runtime-aware loop: no [RUNTIME-AWARE] line yet — wait ~2 minutes and re-run this cell.")
    results.append(("runtime-aware loop passing", None, "pending"))

print()
n_pass = sum(1 for _, ok, _ in results if ok is True)
n_fail = sum(1 for _, ok, _ in results if ok is False)
n_pend = sum(1 for _, ok, _ in results if ok is None)
print(f"Summary: {n_pass} PASS, {n_fail} FAIL, {n_pend} PENDING")
print("Deployment healthy." if n_fail == 0 and n_pend == 0
      else "Deployment not fully healthy yet — see FAIL/PENDING above.")

## Stop (optional)

Stops the socket, the double listener, and the folder watcher. The state
(telemetry, logs, heartbeat) stays in `BASE_DIR` and in Drive.

In [ ]:
stop_socket()
stop_watcher()
stop_folder_watcher()